<a href="https://colab.research.google.com/github/Wasif-13/Flyrank-Internship/blob/main/Copy_of_w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

This notebook builds a simple, auditable baseline action score using two observed signals. The baseline is intended as decision-support and does not use product flags, future-window data, or label-derived inputs.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [ ]:
repo_root = Path("/content/Flyrank-Internship")

print("Repository:", repo_root)
print("Exists:", repo_root.exists())

Repository: /content/Flyrank-Internship
Exists: True


In [ ]:
data_path = repo_root / "data" / "raw" / "content_refresh_anonymized.csv"

print("Data path:", data_path)
print("Exists:", data_path.exists())

df = pd.read_csv(data_path)

print("\nDataset loaded successfully.")
print("Shape:", df.shape)

Data path: /content/Flyrank-Internship/data/raw/content_refresh_anonymized.csv
Exists: True

Dataset loaded successfully.
Shape: (30000, 44)


In [ ]:
print("Columns:")

for i, col in enumerate(df.columns, 1):
    print(f"{i:2}. {col}")

Columns:
 1. content_id
 2. client_id
 3. search_volume
 4. competition
 5. competition_level
 6. cpc
 7. content_type
 8. main_intent
 9. word_count
10. char_count
11. provider_used
12. model_used
13. impressions_90d
14. clicks_90d
15. pageviews_90d
16. sessions_90d
17. users_90d
18. engaged_sessions_90d
19. ai_sessions_90d
20. scroll_events_90d
21. days_with_impressions
22. days_with_sessions
23. impressions_last_30d
24. clicks_last_30d
25. sessions_last_30d
26. impressions_prev_30d
27. clicks_prev_30d
28. sessions_prev_30d
29. content_age_days
30. age_tier
31. age_tier_order
32. days_since_last_update
33. freshness_tier
34. word_count_tier
35. char_count_tier
36. ctr
37. avg_position
38. engagement_rate
39. scroll_rate
40. ai_traffic_pct
41. impression_tier
42. position_tier
43. trend_direction
44. trend_pct


## Signal checks

Before encoding the baseline, I checked two signals that the rule will use. The first is directly linked to refresh-flag reasoning; the second represents observed search demand. Each check includes a visible bucket table with `n` and an evidence-based verdict.

### Signal 1 — Staleness

`days_since_last_update` is the refresh-linked signal. I will check its observed distribution before using it as a directional input to the baseline.

In [ ]:
print("Staleness summary:")
display(df["days_since_last_update"].describe())

print("\nMissing values:")
print(df["days_since_last_update"].isna().sum())

Staleness summary:


,days_since_last_update
count,30000.000000
mean,46.098300
std,42.078709
min,1.000000
25%,20.000000
50%,20.000000
75%,104.000000
max,373.000000



Missing values:
0


In [ ]:
df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[-np.inf, 30, 90, 180, np.inf],
    labels=[
        "0-30 days",
        "31-90 days",
        "91-180 days",
        "181+ days"
    ]
)

staleness_table = (
    df["staleness_bucket"]
    .value_counts(sort=False, dropna=False)
    .rename_axis("staleness_bucket")
    .reset_index(name="n")
)

display(staleness_table)

,staleness_bucket,n
0,0-30 days,20480
1,31-90 days,175
2,91-180 days,9171
3,181+ days,174


**Verdict: CONFIRMED**

The staleness signal is observed across the dataset and provides a usable directional basis for refresh prioritization. The distribution is highly uneven, with most rows in the 0–30 and 91–180 day buckets, so the baseline will use broad staleness bands rather than treating the sparse middle and oldest buckets as equally populated.

### Signal 2 — Search volume

`search_volume` represents observed search demand. I will check its distribution and use simple fixed buckets because the observed values are strongly right-skewed.

In [ ]:
print("Search volume summary:")
display(df["search_volume"].describe())

print("\nMissing values:")
print(df["search_volume"].isna().sum())

Search volume summary:


,search_volume
count,27532.000000
mean,158.882391
std,1518.270825
min,0.000000
25%,0.000000
50%,10.000000
75%,20.000000
max,74000.000000



Missing values:
2468


In [ ]:
df["volume_bucket"] = pd.cut(
    df["search_volume"],
    bins=[-np.inf, 0, 10, 20, 100, np.inf],
    labels=[
        "0",
        "1-10",
        "11-20",
        "21-100",
        "101+"
    ]
)

volume_table = (
    df["volume_bucket"]
    .value_counts(sort=False, dropna=False)
    .rename_axis("volume_bucket")
    .reset_index(name="n")
)

display(volume_table)

,volume_bucket,n
0,0,11081
1,1-10,7311
2,11-20,2290
3,21-100,3801
4,101+,3049
5,NaN,2468


In [ ]:
# Load the existing refresh queue for audit/reference only.
# Its scores, actions, and reason codes will NOT be used as model inputs.

sample_path = repo_root / "outputs" / "refresh_queue_sample.csv"

refresh_sample = pd.read_csv(sample_path)

print("Refresh sample shape:", refresh_sample.shape)

print("\nColumns:")
for i, col in enumerate(refresh_sample.columns, 1):
    print(f"{i:2}. {col}")

Refresh sample shape: (200, 28)

Columns:
 1. final_rank
 2. content_id
 3. client_id
 4. final_refresh_score
 5. best_model_name
 6. best_model_probability
 7. baseline_refresh_score
 8. confidence
 9. suggested_action
10. final_reason_codes
11. is_declining_label
12. impressions_90d
13. clicks_90d
14. sessions_90d
15. avg_position
16. ctr
17. content_age_days
18. days_since_last_update
19. word_count
20. trend_direction
21. competition_level
22. content_type
23. main_intent
24. age_tier
25. freshness_tier
26. word_count_tier
27. impression_tier
28. position_tier


In [ ]:
print("Reference refresh queue preview:")

display(refresh_sample.head())

Reference refresh queue preview:


,final_rank,content_id,client_id,final_refresh_score,best_model_name,best_model_probability,baseline_refresh_score,confidence,suggested_action,final_reason_codes,is_declining_label,impressions_90d,clicks_90d,sessions_90d,avg_position,ctr,content_age_days,days_since_last_update,word_count,trend_direction,competition_level,content_type,main_intent,age_tier,freshness_tier,word_count_tier,impression_tier,position_tier
0,1,content_1f080331fa2b,client_3fdba35f04,81.636697,random_forest,0.782079,0.844481,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|low...,1,12834,6,66,6.8,0.05,165,104,1404.0,down,MEDIUM,keyword article,informational,91-180,91-180,1000-2000,good,page_1
1,2,content_6aa43079fb0c,client_3fdba35f04,81.447656,random_forest,0.788105,0.825477,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,1,8064,6,23,3.8,0.07,139,104,1457.0,down,LOW,keyword article,informational,91-180,91-180,1000-2000,good,page_1
2,3,content_d6570c51c9bd,client_3fdba35f04,81.430346,random_forest,0.847372,0.695884,medium,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,1,2498,0,9,10.1,0.00,165,104,1362.0,down,MEDIUM,keyword article,informational,91-180,91-180,1000-2000,moderate,striking
3,4,content_72e800a9c214,client_3fdba35f04,81.034960,random_forest,0.774371,0.842545,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,1,13790,16,27,8.2,0.12,139,104,1371.0,down,MEDIUM,keyword article,commercial,91-180,91-180,1000-2000,good,page_1
4,5,content_e04eb9549989,client_3fdba35f04,80.873188,random_forest,0.814805,0.749468,medium,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,1,3393,3,5,3.6,0.09,131,104,1408.0,down,LOW,keyword article,informational,91-180,91-180,1000-2000,good,page_1


In [ ]:
required_reference_columns = [
    "content_id",
    "days_since_last_update",
    "search_volume",
    "baseline_refresh_score",
    "suggested_action",
    "final_reason_codes"
]

print("Reference columns available:")

for col in required_reference_columns:
    print(f"{col}: {col in refresh_sample.columns}")

Reference columns available:
content_id: True
days_since_last_update: True
search_volume: False
baseline_refresh_score: True
suggested_action: True
final_reason_codes: True


In [ ]:
# Staleness audit against the reference refresh score

refresh_audit["staleness_bucket"] = pd.cut(
    refresh_audit["days_since_last_update"],
    bins=[-np.inf, 30, 90, 180, np.inf],
    labels=[
        "0-30 days",
        "31-90 days",
        "91-180 days",
        "181+ days"
    ]
)

staleness_audit = (
    refresh_audit
    .groupby("staleness_bucket", observed=False)
    .agg(
        n=("content_id", "size"),
        mean_refresh_score=("baseline_refresh_score", "mean")
    )
    .reset_index()
)

display(staleness_audit)

,staleness_bucket,n,mean_refresh_score
0,0-30 days,65,0.640490
1,31-90 days,0,NaN
2,91-180 days,134,0.708332
3,181+ days,1,0.824268


### Staleness verdict

**CONFIRMED**

Staleness is a real, observed signal and is directly connected to the refresh-review logic. The bucket audit provides directional evidence about how the reference refresh scoring behaves across different levels of days since last update. I will use staleness as one input to the baseline, while treating it as decision-support rather than proof that a page requires refresh.

In [ ]:
# Add the raw signal values to the reference refresh queue.
# The reference queue is used only for auditing the existing logic.

signal_columns = [
    "content_id",
    "days_since_last_update",
    "search_volume"
]

reference_signals = df[signal_columns].copy()

refresh_audit = refresh_sample.merge(
    reference_signals,
    on="content_id",
    how="left",
    suffixes=("", "_raw")
)

print("Refresh audit shape:", refresh_audit.shape)

print("\nMissing signal values after merge:")
print(
    refresh_audit[
        ["days_since_last_update", "search_volume"]
    ].isna().sum()
)

Refresh audit shape: (200, 30)

Missing signal values after merge:
days_since_last_update    0
search_volume             0
dtype: int64


In [ ]:
# Search-volume audit against the reference refresh score

refresh_audit["volume_bucket"] = pd.cut(
    refresh_audit["search_volume"],
    bins=[-np.inf, 0, 10, 20, 100, np.inf],
    labels=[
        "0",
        "1-10",
        "11-20",
        "21-100",
        "101+"
    ]
)

volume_audit = (
    refresh_audit
    .groupby("volume_bucket", observed=False)
    .agg(
        n=("content_id", "size"),
        mean_refresh_score=("baseline_refresh_score", "mean")
    )
    .reset_index()
)

display(volume_audit)

,volume_bucket,n,mean_refresh_score
0,0,111,0.676202
1,1-10,36,0.684600
2,11-20,17,0.703599
3,21-100,19,0.675141
4,101+,17,0.757625


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### Baseline rule

I will rank content for refresh review using two observed signals: staleness and search demand.

The score gives more points to content that has been unchanged for longer and to content with higher observed search volume. The rule is intentionally simple and transparent so it can serve as a Week-4 baseline that a later model can attempt to beat.

The baseline produces exactly one reason code and one action label per row.

Reason codes:

- `STALE_HIGH_DEMAND` — at least 91 days since last update and search volume above 20.
- `STALE_DEMAND` — at least 91 days since last update with search volume of 20 or below.
- `FRESH_HIGH_DEMAND` — less than 91 days since last update and search volume above 20.
- `LOW_PRIORITY` — less than 91 days since last update and search volume of 20 or below.

Actions:

- `REFRESH_REVIEW`
- `MONITOR`

The score is directional decision-support, not a claim that a page definitely requires refresh.

In [ ]:
# Build the Week-4 baseline score.
# Only observed raw signals are used here.

baseline = df.copy()

# ---------------------------------------------------------
# 1. Staleness points
# ---------------------------------------------------------

baseline["staleness_points"] = np.select(
    [
        baseline["days_since_last_update"] >= 181,
        baseline["days_since_last_update"] >= 91,
        baseline["days_since_last_update"] >= 31,
        baseline["days_since_last_update"] >= 0
    ],
    [
        60,
        45,
        20,
        0
    ],
    default=0
)

# ---------------------------------------------------------
# 2. Search-volume points
# ---------------------------------------------------------

baseline["volume_points"] = np.select(
    [
        baseline["search_volume"] >= 101,
        baseline["search_volume"] > 20,
        baseline["search_volume"] > 0,
        baseline["search_volume"] == 0
    ],
    [
        40,
        25,
        10,
        0
    ],
    default=0
)

# ---------------------------------------------------------
# 3. Final score
# ---------------------------------------------------------

baseline["baseline_score"] = (
    baseline["staleness_points"]
    + baseline["volume_points"]
)

# ---------------------------------------------------------
# 4. One reason code
# ---------------------------------------------------------

baseline["reason_code"] = np.select(
    [
        (baseline["days_since_last_update"] >= 91) &
        (baseline["search_volume"] > 20),

        (baseline["days_since_last_update"] >= 91) &
        (baseline["search_volume"] <= 20),

        (baseline["days_since_last_update"] < 91) &
        (baseline["search_volume"] > 20)
    ],
    [
        "STALE_HIGH_DEMAND",
        "STALE_DEMAND",
        "FRESH_HIGH_DEMAND"
    ],
    default="LOW_PRIORITY"
)

# ---------------------------------------------------------
# 5. Action
# ---------------------------------------------------------

baseline["action"] = np.where(
    baseline["baseline_score"] >= 45,
    "REFRESH_REVIEW",
    "MONITOR"
)

print("Baseline created.")
print("Rows:", len(baseline))

display(
    baseline[
        [
            "content_id",
            "days_since_last_update",
            "search_volume",
            "baseline_score",
            "reason_code",
            "action"
        ]
    ]
    .sort_values("baseline_score", ascending=False)
    .head(20)
)

Baseline created.
Rows: 30000


,content_id,days_since_last_update,search_volume,baseline_score,reason_code,action
1659,content_bbca724138f2,236,1600.0,100,STALE_HIGH_DEMAND,REFRESH_REVIEW
23619,content_24abafed9707,231,480.0,100,STALE_HIGH_DEMAND,REFRESH_REVIEW
21984,content_02b0d6e30129,313,110.0,100,STALE_HIGH_DEMAND,REFRESH_REVIEW
15947,content_40e140ba2934,231,720.0,100,STALE_HIGH_DEMAND,REFRESH_REVIEW
20695,content_433e980f7076,104,590.0,85,STALE_HIGH_DEMAND,REFRESH_REVIEW
12163,content_35c45cb034ab,104,320.0,85,STALE_HIGH_DEMAND,REFRESH_REVIEW
16749,content_64e97fb0e71b,104,140.0,85,STALE_HIGH_DEMAND,REFRESH_REVIEW
26442,content_015dd044ee9f,104,260.0,85,STALE_HIGH_DEMAND,REFRESH_REVIEW
20734,content_94153fb48717,104,480.0,85,STALE_HIGH_DEMAND,REFRESH_REVIEW
8407,content_d1e915d03c28,104,140.0,85,STALE_HIGH_DEMAND,REFRESH_REVIEW


In [ ]:
# Rank the complete queue.

baseline = baseline.sort_values(
    [
        "baseline_score",
        "search_volume",
        "days_since_last_update"
    ],
    ascending=[False, False, False]
).reset_index(drop=True)

baseline["rank"] = np.arange(1, len(baseline) + 1)

print("Ranked rows:", len(baseline))

display(
    baseline[
        [
            "rank",
            "content_id",
            "baseline_score",
            "reason_code",
            "action",
            "days_since_last_update",
            "search_volume"
        ]
    ].head(20)
)

Ranked rows: 30000


,rank,content_id,baseline_score,reason_code,action,days_since_last_update,search_volume
0,1,content_bbca724138f2,100,STALE_HIGH_DEMAND,REFRESH_REVIEW,236,1600.0
1,2,content_40e140ba2934,100,STALE_HIGH_DEMAND,REFRESH_REVIEW,231,720.0
2,3,content_24abafed9707,100,STALE_HIGH_DEMAND,REFRESH_REVIEW,231,480.0
3,4,content_02b0d6e30129,100,STALE_HIGH_DEMAND,REFRESH_REVIEW,313,110.0
4,5,content_ef99c4abd9ab,85,STALE_HIGH_DEMAND,REFRESH_REVIEW,104,74000.0
5,6,content_bf67a444faef,85,STALE_HIGH_DEMAND,REFRESH_REVIEW,104,60500.0
6,7,content_5ec29ae79c60,85,STALE_HIGH_DEMAND,REFRESH_REVIEW,104,60500.0
7,8,content_deb54e9e19cd,85,STALE_HIGH_DEMAND,REFRESH_REVIEW,104,60500.0
8,9,content_454cc6654c6e,85,STALE_HIGH_DEMAND,REFRESH_REVIEW,104,60500.0
9,10,content_7868341d97dd,85,STALE_HIGH_DEMAND,REFRESH_REVIEW,104,40500.0


# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

The queue ranks every row using the baseline score. The output contains the rank, score, single reason code, and action label required for downstream review.

In [ ]:
output_dir = repo_root / "work" / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "baseline_action_score.csv"

queue_columns = [
    "rank",
    "content_id",
    "baseline_score",
    "reason_code",
    "action"
]

baseline[queue_columns].to_csv(
    output_path,
    index=False
)

print("Output:", output_path)
print("Exists:", output_path.exists())
print("Rows written:", len(baseline))

display(pd.read_csv(output_path).head(20))

Output: /content/Flyrank-Internship/work/outputs/baseline_action_score.csv
Exists: True
Rows written: 30000


,rank,content_id,baseline_score,reason_code,action
0,1,content_bbca724138f2,100,STALE_HIGH_DEMAND,REFRESH_REVIEW
1,2,content_40e140ba2934,100,STALE_HIGH_DEMAND,REFRESH_REVIEW
2,3,content_24abafed9707,100,STALE_HIGH_DEMAND,REFRESH_REVIEW
3,4,content_02b0d6e30129,100,STALE_HIGH_DEMAND,REFRESH_REVIEW
4,5,content_ef99c4abd9ab,85,STALE_HIGH_DEMAND,REFRESH_REVIEW
5,6,content_bf67a444faef,85,STALE_HIGH_DEMAND,REFRESH_REVIEW
6,7,content_5ec29ae79c60,85,STALE_HIGH_DEMAND,REFRESH_REVIEW
7,8,content_deb54e9e19cd,85,STALE_HIGH_DEMAND,REFRESH_REVIEW
8,9,content_454cc6654c6e,85,STALE_HIGH_DEMAND,REFRESH_REVIEW
9,10,content_7868341d97dd,85,STALE_HIGH_DEMAND,REFRESH_REVIEW


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

The top 20 are reviewed as decision-support recommendations. Each row includes the action, single reason code, a confidence note, and a specific condition that could make the recommendation wrong.

In [ ]:
top20 = baseline.head(20).copy()

top20["confidence_note"] = np.select(
    [
        top20["baseline_score"] >= 85,
        top20["baseline_score"] >= 45
    ],
    [
        "Higher baseline confidence because both staleness and demand contribute strongly.",
        "Moderate baseline confidence based on the observed rule."
    ],
    default="Lower baseline confidence because fewer signals contribute."
)

top20["what_would_make_it_wrong"] = np.select(
    [
        top20["reason_code"] == "STALE_HIGH_DEMAND",
        top20["reason_code"] == "STALE_DEMAND",
        top20["reason_code"] == "FRESH_HIGH_DEMAND"
    ],
    [
        "The page may already have been refreshed outside the available snapshot or may not benefit from another refresh.",
        "The page may be intentionally stable and its lower demand may not justify refresh effort.",
        "The page may already perform well and high demand alone may not justify a refresh.",
    ],
    default="The simple baseline may miss an opportunity caused by signals not included in this rule."
)

top20_review = top20[
    [
        "rank",
        "content_id",
        "action",
        "reason_code",
        "baseline_score",
        "confidence_note",
        "what_would_make_it_wrong"
    ]
].copy()

display(top20_review)

,rank,content_id,action,reason_code,baseline_score,confidence_note,what_would_make_it_wrong
0,1,content_bbca724138f2,REFRESH_REVIEW,STALE_HIGH_DEMAND,100,Higher baseline confidence because both stalen...,The page may already have been refreshed outsi...
1,2,content_40e140ba2934,REFRESH_REVIEW,STALE_HIGH_DEMAND,100,Higher baseline confidence because both stalen...,The page may already have been refreshed outsi...
2,3,content_24abafed9707,REFRESH_REVIEW,STALE_HIGH_DEMAND,100,Higher baseline confidence because both stalen...,The page may already have been refreshed outsi...
3,4,content_02b0d6e30129,REFRESH_REVIEW,STALE_HIGH_DEMAND,100,Higher baseline confidence because both stalen...,The page may already have been refreshed outsi...
4,5,content_ef99c4abd9ab,REFRESH_REVIEW,STALE_HIGH_DEMAND,85,Higher baseline confidence because both stalen...,The page may already have been refreshed outsi...
5,6,content_bf67a444faef,REFRESH_REVIEW,STALE_HIGH_DEMAND,85,Higher baseline confidence because both stalen...,The page may already have been refreshed outsi...
6,7,content_5ec29ae79c60,REFRESH_REVIEW,STALE_HIGH_DEMAND,85,Higher baseline confidence because both stalen...,The page may already have been refreshed outsi...
7,8,content_deb54e9e19cd,REFRESH_REVIEW,STALE_HIGH_DEMAND,85,Higher baseline confidence because both stalen...,The page may already have been refreshed outsi...
8,9,content_454cc6654c6e,REFRESH_REVIEW,STALE_HIGH_DEMAND,85,Higher baseline confidence because both stalen...,The page may already have been refreshed outsi...
9,10,content_7868341d97dd,REFRESH_REVIEW,STALE_HIGH_DEMAND,85,Higher baseline confidence because both stalen...,The page may already have been refreshed outsi...


In [ ]:
print("Top-20 reason codes:")
display(
    top20_review["reason_code"]
    .value_counts()
    .rename_axis("reason_code")
    .reset_index(name="n")
)

print("\nTop-20 actions:")
display(
    top20_review["action"]
    .value_counts()
    .rename_axis("action")
    .reset_index(name="n")
)

Top-20 reason codes:


,reason_code,n
0,STALE_HIGH_DEMAND,20



Top-20 actions:


,action,n
0,REFRESH_REVIEW,20


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### Weak picks

The baseline is intentionally simple, so some picks may be weak. A high score does not prove that a page needs a refresh. Potential weak picks include content that is old but intentionally stable, pages with high search demand that already perform well, and pages where important context is not represented by the two baseline signals.

The main limitation is that this baseline uses only staleness and search demand, so it can miss other useful signals.

In [ ]:
# Show lower-confidence examples from the ranked queue.

weak_candidates = baseline[
    baseline["reason_code"].isin(
        ["STALE_DEMAND", "FRESH_HIGH_DEMAND", "LOW_PRIORITY"]
    )
].head(10)

display(
    weak_candidates[
        [
            "rank",
            "content_id",
            "baseline_score",
            "reason_code",
            "action",
            "days_since_last_update",
            "search_volume"
        ]
    ]
)

,rank,content_id,baseline_score,reason_code,action,days_since_last_update,search_volume
1925,1926,content_dd413158df3c,70,STALE_DEMAND,REFRESH_REVIEW,305,20.0
1926,1927,content_6476d1d8c050,70,STALE_DEMAND,REFRESH_REVIEW,313,10.0
1927,1928,content_94991fe6268c,70,STALE_DEMAND,REFRESH_REVIEW,313,10.0
1928,1929,content_026a1e2a82fd,70,STALE_DEMAND,REFRESH_REVIEW,305,10.0
1929,1930,content_ccf25ed65a99,70,STALE_DEMAND,REFRESH_REVIEW,305,10.0
1930,1931,content_d25a099b3726,70,STALE_DEMAND,REFRESH_REVIEW,305,10.0
1931,1932,content_129753e3095f,70,STALE_DEMAND,REFRESH_REVIEW,305,10.0
1932,1933,content_f2b4acf220d9,70,STALE_DEMAND,REFRESH_REVIEW,305,10.0
1933,1934,content_afd26a07382d,70,STALE_DEMAND,REFRESH_REVIEW,305,10.0
1934,1935,content_ab18b5811c02,70,STALE_DEMAND,REFRESH_REVIEW,305,10.0


In [ ]:
# Explicit leakage check

score_inputs = {
    "days_since_last_update",
    "search_volume"
}

forbidden_reference_outputs = {
    "baseline_refresh_score",
    "final_refresh_score",
    "suggested_action",
    "final_reason_codes"
}

print("Inputs used by baseline:")
for col in sorted(score_inputs):
    print(" -", col)

print("\nReference/product outputs NOT used:")
for col in sorted(forbidden_reference_outputs):
    print(" -", col)

print("\nLeakage checks:")
print("Product flags used in score: NO")
print("Existing refresh scores used in score: NO")
print("Existing actions used in score: NO")
print("Existing reason codes used in score: NO")
print("Future-window inputs used: NO")
print("Label-derived inputs used: NO")

Inputs used by baseline:
 - days_since_last_update
 - search_volume

Reference/product outputs NOT used:
 - baseline_refresh_score
 - final_reason_codes
 - final_refresh_score
 - suggested_action

Leakage checks:
Product flags used in score: NO
Existing refresh scores used in score: NO
Existing actions used in score: NO
Existing reason codes used in score: NO
Future-window inputs used: NO
Label-derived inputs used: NO


In [ ]:
print("FINAL OUTPUT CHECK")
print("=" * 50)

saved_queue = pd.read_csv(output_path)

print("CSV exists:", output_path.exists())
print("CSV shape:", saved_queue.shape)
print("Expected rows:", len(df))

print("\nColumns:")
print(saved_queue.columns.tolist())

print("\nReason codes:")
print(saved_queue["reason_code"].value_counts())

print("\nActions:")
print(saved_queue["action"].value_counts())

print("\nTop 10:")
display(saved_queue.head(10))

FINAL OUTPUT CHECK
CSV exists: True
CSV shape: (30000, 5)
Expected rows: 30000

Columns:
['rank', 'content_id', 'baseline_score', 'reason_code', 'action']

Reason codes:
reason_code
LOW_PRIORITY         16137
STALE_DEMAND          7013
FRESH_HIGH_DEMAND     4925
STALE_HIGH_DEMAND     1925
Name: count, dtype: int64

Actions:
action
MONITOR           20608
REFRESH_REVIEW     9392
Name: count, dtype: int64

Top 10:


,rank,content_id,baseline_score,reason_code,action
0,1,content_bbca724138f2,100,STALE_HIGH_DEMAND,REFRESH_REVIEW
1,2,content_40e140ba2934,100,STALE_HIGH_DEMAND,REFRESH_REVIEW
2,3,content_24abafed9707,100,STALE_HIGH_DEMAND,REFRESH_REVIEW
3,4,content_02b0d6e30129,100,STALE_HIGH_DEMAND,REFRESH_REVIEW
4,5,content_ef99c4abd9ab,85,STALE_HIGH_DEMAND,REFRESH_REVIEW
5,6,content_bf67a444faef,85,STALE_HIGH_DEMAND,REFRESH_REVIEW
6,7,content_5ec29ae79c60,85,STALE_HIGH_DEMAND,REFRESH_REVIEW
7,8,content_deb54e9e19cd,85,STALE_HIGH_DEMAND,REFRESH_REVIEW
8,9,content_454cc6654c6e,85,STALE_HIGH_DEMAND,REFRESH_REVIEW
9,10,content_7868341d97dd,85,STALE_HIGH_DEMAND,REFRESH_REVIEW


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.